In [4]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from google.genai import types

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

In [ ]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [7]:
gemini_model = "gemini-3.5-flash-lite"
groq_model = "openai/gpt-oss-120b"
ollama_model = 'llama3.2'

# Define personalities
system_prompts = {
    "Alex": "You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and challenge everything in a snarky way. Keep your responses short and punchy.",
    "Blake": "You are Blake, an overly optimistic chatbot who looks for the silver lining in absolutely everything. You are polite and enthusiastic.",
    "Charlie": "You are Charlie, a highly analytical, strictly data-driven chatbot who hates emotion and only cares about facts and logic."
}

# Initialize master history
conversation_history = [
    "Charlie: I have benchmarked our frontend application, and utilizing raw inline Tailwind utility classes is statistically proven to reduce our bundle asset size and optimize DOM rendering efficiency compared to maintaining separate CSS modules."
]

def build_user_prompt(persona, history_text):
    return f"""
You are {persona}, in conversation with the other agents.
The conversation so far is as follows:
{history_text}

Now with this, respond with what you would like to say next, as {persona}. Do not prefix your response with your name.
"""

In [8]:
# Client 1: Gemini (Alex)
def call_alex(history_text):
    response = gemini.chat.completions.create(
        model=gemini_model,
        messages=[
            {"role": "system", "content": system_prompts["Alex"]},
            {"role": "user", "content": build_user_prompt("Alex", history_text)}
        ]
    )
    return response.choices[0].message.content.strip() 

# Client 2: Groq (Blake)
def call_blake(history_text):
    response = groq.chat.completions.create(
        model=groq_model,
        messages=[
            {"role": "user", "content": build_user_prompt("Blake", history_text)}
        ]
    )
    return response.choices[0].message.content.strip()

# Client 3: Ollama Native Client (Charlie)
def call_charlie(history_text):
    response = ollama.chat.completions.create(
        model=ollama_model,
        messages=[
            {"role": "system", "content": system_prompts["Charlie"]},
            {"role": "user", "content": build_user_prompt("Charlie", history_text)}
        ]
    )
    return response.choices[0].message.content.strip()

In [ ]:
# Master routing dictionary to map the persona to their unique client runner
agent_runners = {
    "Alex": call_alex,
    "Blake": call_blake,
    "Charlie": call_charlie
}

# 3. Run the conversation loop
turns = ["Alex", "Blake", "Charlie", "Alex", "Blake", "Charlie"]

display(Markdown("### --- Multi-Client 3-Way Conversation Started ---\n"))
display(Markdown(f"### Charlie: \n{conversation_history[0]}\n"))


for current_agent in turns:
    full_transcript = "\n".join(conversation_history)
    
    # Dynamically call the specific client function for this agent
    reply = agent_runners[current_agent](full_transcript)
    
    formatted_reply = f"{current_agent}: {reply}\n"
    conversation_history.append(formatted_reply)
    
    display(Markdown(f"### {current_agent}:\n {reply}\n"))